# T04: Healthcare Domain Deep Dive

## What You'll Learn

Spindle ships with a **Healthcare** domain model that generates realistic clinical and
financial data: patients, encounters, diagnoses (ICD-10), procedures, medications, and
claims.  This tutorial walks through the generated data table by table so you understand
what you're getting and why the distributions look the way they do.

If you work in **Revenue Cycle Management (RCM)**, **EHR analytics**, or **claims
processing**, this is your starting point.

## Tables in the Healthcare Domain

| Table | Purpose |
|---|---|
| `provider` | Physicians and clinicians |
| `facility` | Hospitals, clinics, labs |
| `patient` | Demographics, insurance |
| `encounter` | Office visits, ED visits, admissions |
| `diagnosis` | ICD-10 codes linked to encounters |
| `procedure` | CPT/HCPCS codes linked to encounters |
| `medication` | Prescriptions linked to encounters |
| `claim` | Payer claims tied to encounters |
| `claim_line` | Line-item detail for each claim |

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with `Spindle.generate()` (see **T01** or **T02**)

## Time Estimate

**~15 minutes**

## Step 1 — Generate the Healthcare Dataset

**What we're about to do:** Import Spindle and the `HealthcareDomain`, then generate
a small-scale healthcare dataset.

**Why this matters:** The `HealthcareDomain` encapsulates all nine tables, their
relationships, realistic value distributions (Census demographics, CMS diagnosis
frequencies, payer mix), and referential integrity — all in a single call.

**What to expect:** A summary listing every table and its row count.

In [ ]:
from sqllocks_spindle import Spindle, HealthcareDomain

spindle = Spindle(HealthcareDomain)
data = spindle.generate(scale="small")

print("Healthcare Domain — Small Scale")
print("=" * 40)
for table_name, df in data.items():
    print(f"  {table_name:<15} {len(df):>6} rows")
print(f"\nTotal tables: {len(data)}")

## Step 2 — Patient Demographics

**What we're about to do:** Examine the `patient` table's gender and race/ethnicity
distributions.

**Why this matters:** Spindle's healthcare domain uses **U.S. Census-based distributions**
for demographic fields.  This means the generated data reflects real-world population
proportions — essential for realistic analytics, dashboard prototyping, and ML model
testing where demographic skew can introduce bias.

**What to expect:** Gender split and race/ethnicity breakdown with percentages that
approximate Census data.

In [ ]:
patient = data["patient"]

print("Patient Gender Distribution")
print("=" * 35)
gender = patient["gender"].value_counts()
gender_pct = patient["gender"].value_counts(normalize=True).mul(100).round(1)
for val in gender.index:
    print(f"  {val:<12} {gender[val]:>5} patients  ({gender_pct[val]}%)")

print("\nPatient Race/Ethnicity Distribution")
print("=" * 45)
race = patient["race"].value_counts()
race_pct = patient["race"].value_counts(normalize=True).mul(100).round(1)
for val in race.index:
    print(f"  {val:<30} {race[val]:>5} patients  ({race_pct[val]}%)")

## Step 3 — Encounter Types

**What we're about to do:** Look at the distribution of encounter types in the
`encounter` table.

**Why this matters:** In the U.S. healthcare system, encounter types follow a well-known
pattern:

- **~70% Outpatient** — routine office visits, follow-ups
- **~10% Emergency Department (ED)** — urgent/emergent care
- **~10% Inpatient** — hospital admissions
- **~10% Telehealth** — virtual visits (growing post-2020)

Spindle's healthcare domain encodes these proportions so your test data mirrors
real utilization patterns.

**What to expect:** A count and percentage for each encounter type, with outpatient
dominating.

In [ ]:
encounter = data["encounter"]

print("Encounter Type Distribution")
print("=" * 45)
enc_type = encounter["encounter_type"].value_counts()
enc_pct = encounter["encounter_type"].value_counts(normalize=True).mul(100).round(1)
for val in enc_type.index:
    print(f"  {val:<15} {enc_type[val]:>6} encounters  ({enc_pct[val]}%)")
print(f"\nTotal encounters: {len(encounter)}")

## Step 4 — Diagnoses and ICD-10 Codes

**What we're about to do:** Examine the `diagnosis` table — specifically the ICD-10
codes and diagnosis types it contains.

**Why this matters:** Spindle generates ICD-10 codes following **realistic CMS
distribution patterns**.  The most common codes (essential hypertension, type 2 diabetes,
hyperlipidemia) appear frequently, while rare conditions appear infrequently.  Diagnosis
types (primary, secondary, admitting) are also distributed realistically.

This is critical for building claims analytics, DRG groupers, or risk-adjustment models
where code frequency matters.

**What to expect:** The first few rows of the diagnosis table, plus a breakdown of
diagnosis types.

In [ ]:
diagnosis = data["diagnosis"]

print("Diagnosis Table — First 10 Rows")
print(diagnosis.head(10).to_string(index=False))

print("\nDiagnosis Type Distribution")
print("=" * 40)
dx_type = diagnosis["diagnosis_type"].value_counts()
dx_pct = diagnosis["diagnosis_type"].value_counts(normalize=True).mul(100).round(1)
for val in dx_type.index:
    print(f"  {val:<15} {dx_type[val]:>6} diagnoses  ({dx_pct[val]}%)")
print(f"\nUnique ICD-10 codes: {diagnosis['icd10_code'].nunique()}")

## Step 5 — Claims: Status Distribution and Denial Rate

**What we're about to do:** Inspect the `claim` table's status column to see the
distribution of paid, denied, and pending claims.

**Why this matters:** Denial rate is one of the most important KPIs in Revenue Cycle
Management.  Industry benchmarks put the average denial rate at **5–10%**, with some
organizations seeing rates above 15%.  Spindle's healthcare domain generates claim
statuses that reflect a realistic payer adjudication mix, giving you meaningful data
for denial analytics dashboards and A/R aging reports.

**What to expect:** A count of claims by status, plus the computed denial rate as a
percentage.

In [ ]:
claim = data["claim"]

print("Claim Status Distribution")
print("=" * 40)
status = claim["status"].value_counts()
status_pct = claim["status"].value_counts(normalize=True).mul(100).round(1)
for val in status.index:
    print(f"  {val:<12} {status[val]:>6} claims  ({status_pct[val]}%)")

denied_count = status.get("denied", 0)
denial_rate = denied_count / len(claim) * 100
print(f"\nDenial rate: {denial_rate:.1f}%")
print(f"Total claims: {len(claim)}")

## Step 6 — The Full Clinical Flow: Encounter → Diagnosis → Claim

**What we're about to do:** Join the `encounter`, `diagnosis`, and `claim` tables to
trace a single patient's clinical journey from visit to billing.

**Why this matters:** In real healthcare data, the clinical flow looks like this:

1. A **patient** arrives at a **facility** and sees a **provider** → an **encounter** is created
2. The provider documents **diagnoses** (ICD-10) and **procedures** (CPT)
3. A **claim** is submitted to the payer with **claim lines** for each service

Spindle preserves this referential chain, so JOINs across the model work exactly
as they would on production data.  This is essential for building realistic ETL
pipelines, data models, and analytics.

**What to expect:** A merged DataFrame showing one patient's encounters, their
diagnoses, and the corresponding claim statuses.

In [ ]:
import pandas as pd

# Pick the patient with the most encounters for a rich example
encounter = data["encounter"]
top_patient_id = encounter["patient_id"].value_counts().index[0]

# Filter to that patient's encounters
pat_encounters = encounter[encounter["patient_id"] == top_patient_id]

# Join diagnosis
pat_dx = pat_encounters.merge(
    data["diagnosis"], on="encounter_id", how="left"
)

# Join claim
pat_full = pat_dx.merge(
    data["claim"][["encounter_id", "status", "total_charge"]],
    on="encounter_id",
    how="left",
)

display_cols = [
    "patient_id", "encounter_id", "encounter_type",
    "icd10_code", "diagnosis_type", "status", "total_charge",
]
# Keep only columns that exist (some merges may vary)
display_cols = [c for c in display_cols if c in pat_full.columns]

print(f"Clinical Journey for Patient {top_patient_id}")
print("=" * 80)
print(pat_full[display_cols].head(15).to_string(index=False))
print(f"\nTotal rows for this patient: {len(pat_full)}")

## What's Next

You have explored the Healthcare domain end-to-end: demographics, encounters,
diagnoses, and claims.  Every table is referentially consistent and statistically
realistic.

### Recommended next steps

| Notebook / Resource | What You'll Learn |
|---|---|
| **F07 — Healthcare RCM Notebook** | Build a full Revenue Cycle Management dashboard from this data |
| **Methodology Page** | How Spindle derives its Census, CMS, and payer-mix distributions |

### Quick links

- [F07_healthcare_rcm.ipynb](../focused/F07_healthcare_rcm.ipynb)
- [Methodology](../../reference/methodology.md)